In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
%%capture
!pip install -q transformers==4.45.0 trl==0.11.4 peft==0.13.2 accelerate==0.34.2 sentencepiece datasets
!pip install -q bitsandbytes --upgrade

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DATA_DIR   = '/content/drive/MyDrive/LegalRAG/cleaned_data'
OUTPUT_DIR = '/tmp/llm_ft_context'

KAGGLE_TRAIN = os.path.join(DATA_DIR, 'kaggle_train_clean.jsonl')
KAGGLE_TEST  = os.path.join(DATA_DIR, 'kaggle_test_clean.jsonl')

os.makedirs(OUTPUT_DIR, exist_ok=True)

for path in [KAGGLE_TRAIN, KAGGLE_TEST]:
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    print(f"{'✅' if exists else '❌'} {os.path.basename(path)} ({size:.1f} MB)")

In [ ]:
import json, random
from datasets import Dataset
import numpy as np

SYSTEM_PROMPT = (
    "Sen bir Türk hukuku uzmanı asistanısın. "
    "Cevaplarını SADECE verilen kanun metnine dayandır. "
    "Kanun metninde olmayan bilgileri ekleme. "
    "Her cevabın sonunda ilgili kanun maddesini belirt."
)

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def to_llama3_chat(record):
    soru    = record.get('soru', '').strip()
    cevap   = record.get('cevap', '').strip()
    context = record.get('context', '').strip()

    if context:
        user_content = f"Aşağıdaki kanun maddesine dayanarak soruyu yanıtla:\n\n{context}\n\nSoru: {soru}"
    else:
        user_content = soru

    text = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{cevap}<|eot_id|>"
    )
    return {'text': text}

print('Yükleniyor...')
kaggle_train_raw = load_jsonl(KAGGLE_TRAIN)
kaggle_test_raw  = load_jsonl(KAGGLE_TEST)

print(f'Kaggle train: {len(kaggle_train_raw):,}')
print(f'Kaggle test:  {len(kaggle_test_raw):,}')

context_var = sum(1 for r in kaggle_train_raw if r.get('context', '').strip())
print(f'Context olan: {context_var:,}')

random.seed(42)
random.shuffle(kaggle_train_raw)

train_formatted = [to_llama3_chat(r) for r in kaggle_train_raw]
val_formatted   = [to_llama3_chat(r) for r in kaggle_test_raw]

print(f'\nToplam train: {len(train_formatted):,}')
print(f'Toplam val:   {len(val_formatted):,}')

lengths = [len(r['text'].split()) for r in train_formatted]
print(f'Ortalama uzunluk: {np.mean(lengths):.0f} word')
print(f'90. persantil:    {np.percentile(lengths, 90):.0f} word')

print('\n--- Örnek ---')
print(train_formatted[0]['text'][:600])

In [ ]:
train_dataset = Dataset.from_list(train_formatted)
val_dataset   = Dataset.from_list(val_formatted)
print(train_dataset)
print(val_dataset)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID    = 'Trendyol/Trendyol-LLM-8b-chat-v2.0'
MAX_SEQ_LEN = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print('Tokenizer yükleniyor...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Model yükleniyor (4-bit)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

print(f'Model yüklendi. Parametreler: {model.num_parameters()/1e9:.2f}B')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    bf16=True,
    fp16=False,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='no',
    max_grad_norm=0.3,
    report_to='none',
    seed=42,
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

print(f'Train steps/epoch: {len(trainer.get_train_dataloader())}')

In [ ]:
import time

print('Eğitim başlıyor...')
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f'Eğitim tamamlandı. Süre: {elapsed/3600:.2f} saat')

In [ ]:
ft_model = trainer.model
ft_model.eval()

def generate_answer(soru, context=None, max_new_tokens=256):
    if context:
        user_content = f"Aşağıdaki kanun maddesine dayanarak soruyu yanıtla:\n\n{context}\n\nSoru: {soru}"
    else:
        user_content = soru

    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(ft_model.device)
    with torch.no_grad():
        output = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

test_sorular = [
    'Türk Ceza Kanununa göre kasten öldürme suçunun cezası nedir?',
    'İş sözleşmesi hangi hallerde haklı nedenle feshedilebilir?',
]

for soru in test_sorular:
    print(f'\n{"="*60}')
    print(f'SORU: {soru}')
    print(f'CEVAP: {generate_answer(soru)}')

In [ ]:
# Model hâlâ var mı kontrol et
try:
    print(type(trainer.model))
    print("✅ Model RAM'de")
except:
    print("❌ Model kaybolmuş, yeniden eğitmek lazım")

In [ ]:
!pip install transformers --upgrade -q

In [ ]:
adapter_path = '/content/drive/MyDrive/LegalRAG/models/trendyol_ft_context'
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print("✅ Kaydedildi")